In [84]:
# import modules
import os, sys, time, random, yaml, re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.rich import tqdm
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter
import matplotlib.ticker as ticker
from collections import Counter
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from typing import List, Dict
from datetime import timedelta

# Set up plot defaults
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = 14.0,10.0  # Roughly 11 cm wde by 8 cm high
mpl.rcParams['font.size'] = 20.0 # Use 14 point font
sns.set(style="whitegrid")

font_size = {
    "xlabel": 17,
    "ylabel": 17,
    "xticks": 15,
    "yticks": 15,
    "legend": 13,
    "title": 13,
}

plt.rcParams.update({
    "axes.labelsize": font_size["xlabel"],  # X and Y axis labels
    "xtick.labelsize": font_size["xticks"],  # X ticks
    "ytick.labelsize": font_size["yticks"],  # Y ticks
    "legend.fontsize": font_size["legend"]  # Legend
})

# loading the yaml file
with open('metadata.key_variables.arm.yml', 'r') as file:
    yml = yaml.safe_load(file)

In [ ]:
# Trim and save the csv to itself with only the specified columns
for site in yml['NSA'].keys():
    for instrument in yml['NSA'][site].keys():
        filename = yml['NSA'][site][instrument]['filename']
        var_list = yml['NSA'][site][instrument]['var_name']

        try:
            df = pd.read_csv(f"../../data/{filename}", usecols=var_list)
            df.to_csv(f"../data/{filename}", index=False)
            print(f"✅ Trimmed and saved: {filename}")

        except Exception as e:
            print(f"❌ Failed to process {filename}: {e}")


In [ ]:
fontfamily = "Open Sans"
dataset = 'SRS'

data = yml['NSA']['C1'][dataset]
vars = data['var_name']
title = data['title']
filename = data['filename']

# Data Preparation
df = pd.read_csv(f"../data/{filename}", usecols=vars)
df[vars[0]] = pd.to_datetime(df[vars[0]])
df = df.set_index(vars[0])
df = df.resample("10T").mean().dropna().reset_index()
df['week'] = (df[vars[0]] - df[vars[0]].min()).dt.days // 7 + 1

groups = {
    "Air Temperature": [v for v in vars if "air_temperature" in v and v != vars[0]],
    "Distance": [v for v in vars if "distance_" in v],
    "Data Quality": [v for v in vars if "data_quality_" in v],
}

trace_meta = {}
figs = {}

for group_name, group_vars in groups.items():
    fig = go.Figure()
    trace_meta[group_name] = []

    for var in group_vars:
        for week in range(1, 5):
            df_week = df[df['week'] == week]
            trace = go.Scattergl(
                x=df_week[vars[0]],
                y=df_week[var],
                mode='lines',
                name=f"{var} (Week {week})",
                visible=(week == 1)
            )
            fig.add_trace(trace)
            trace_meta[group_name].append(week)

    fig.update_layout(
        paper_bgcolor='#B5828C', 
        title=dict(text=f"{title} - {group_name} (Week 1)",
                    font=dict(color='#EBFDFB', size=26, family=fontfamily)),
        xaxis=dict(title=dict(text=vars[0], font=dict(color='#EBFDFB', size=20, family=fontfamily)),
                   tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)),
        yaxis=dict(title=dict(text=group_name, font=dict(color='#EBFDFB', size=20, family=fontfamily)),
                   tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)),
        legend=dict(
            x=1.02, y=1, yanchor='top',
            # bordercolor="white", borderwidth=1,
            font=dict(color='#EBFDFB', size=15, family=fontfamily)
        ),
        updatemenus=[dict(
            type="dropdown",
            direction="down",
            showactive=True,
            x=1.02,
            y=1.1,
            xanchor="left",
            yanchor="top",
            font=dict(color="lightgrey", size=15, family=fontfamily),
            buttons=[
                dict(
                    label=f"Week {w}",
                    method="update",
                    args=[
                        {"visible": [t == w for t in trace_meta[group_name]]},
                        {"title": f"{title} - {group_name} (Week {w})"}
                    ],
                )
                for w in range(1, 5)
            ]
        )],
        height=800
    )

    figs[group_name] = fig
    fig.show()

html_blocks = []
for i, (name, fig) in enumerate(figs.items()):
    fig_html = pio.to_html(fig, include_plotlyjs=('cdn' if i == 0 else False), full_html=False)
    block = f"""
    <div style="display: flex; justify-content: center; margin-top: 20px;">
        <div id="{name.replace(' ', '_').lower()}_container" style="width: 90%;">
            {fig_html}
        </div>
    </div>
    """
    html_blocks.append(block)

full_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>SRS Plotly Plots</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.0/css/all.min.css">
    <link rel="stylesheet" href="/style.css">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
</head>
<body>
    <h1>SRS Plotly Plots</h1>

    <nav id="cut-nav"> 
        <ul id="cut-links">
            <li><a href="SRS.html" class="active">SRS</a></li>
            <li><a href="LPM.html">LPM</a></li>
            <li><a href="SPMF.html">SPMF</a></li>
            <li><a href="SONIC.html">SONIC</a></li>
        </ul>
    </nav>

    <!-- performance or selection plots -->
    <nav id="mode-nav">
    </nav>

    <!-- Section to display images -->
    <section>
        <h2 id="cut-title">{title} Description<br>
            Can drag to zoom in & double click to reset below images
        </h2>
        <div class="wrapper" id="image-container">
            <!-- Images will be dynamically added here -->
        </div>
    </section>

    <!-- Modal for Fullscreen Image -->
    <div id="image-modal" class="modal">
        <span class="close">&times;</span>
        <img class="modal-content" id="modal-img">
    </div>


    <!-- Sidebar -->
    <div id="sidebar">
    <h3>Test Plots</h3>
    <ul>
        <li><a href="../index.html#main" id="main-btn">main</a></li>
        <li><a href="SRS.html" id="test-btn">SRS</a></li>
        <li><a href="LPM.html" id="test-btn">LPM</a></li>
        <li><a href="SPMF.html" id="test-btn">SPMF</a></li>
        <li><a href="SONIC.html" id="test-btn">SONIC</a></li>
    </ul>
    </div>

    <div id="sidebar-toggle"><i class="fas fa-angle-right"></i></div>

    <!-- Plotly Graphs -->
  {''.join(html_blocks)}
    <script src="/sidebar.js"></script>


</body>
</html>
"""

# Save to file
with open("tmp1.html", "w", encoding="utf-8") as f:
    f.write(full_html)

In [ ]:
fontfamily = "Open Sans"
dataset = 'SPMF'

data = yml['NSA']['C1'][dataset]
vars = data['var_name']
title = data['title']
filename = data['filename']

# Data Preparation
df = pd.read_csv(f"../data/{filename}", usecols=vars)
df[vars[0]] = pd.to_datetime(df[vars[0]])
df = df.set_index(vars[0])
df = df.resample("10T").mean().dropna().reset_index()
df['week'] = (df[vars[0]] - df[vars[0]].min()).dt.days // 7 + 1

trace_meta = {}
figs = {}

for var in vars[1:]:
    fig = go.Figure()
    trace_meta[var] = []

    for week in range(1, 5):
        df_week = df[df['week'] == week]
        trace = go.Scattergl(
            x=df_week[vars[0]],
            y=df_week[var],
            mode='lines',
            name=f"{var} (Week {week})",
            visible=(week==1),
            showlegend=True,
            line=dict(color="#55AD9B")
        )
        fig.add_trace(trace)

    fig.update_layout(
        paper_bgcolor='#B5828C', 
        # plot_bgcolor='#B5828C',
        title=dict(text=f"{title} - {var} (Week 1)",
                    font=dict(color='#EBFDFB', size=26, family=fontfamily)),
        xaxis=dict(title=dict(text=vars[0], font=dict(color='#EBFDFB', size=20, family=fontfamily)),
                   tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)),
        yaxis=dict(title=dict(text=var, font=dict(color='#EBFDFB', size=20, family=fontfamily)),
                   tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)),
        legend=dict(
            x=1.02, y=1, yanchor='top',
            # bordercolor="white", borderwidth=1,
            font=dict(color='#EBFDFB', size=15, family=fontfamily)
        ),
        updatemenus=[dict(
            type="dropdown",
            direction="down",
            showactive=True,
            x=1.02,
            y=1.1,
            xanchor="left",
            yanchor="top",
            font=dict(color="lightgrey", size=15, family=fontfamily),
            buttons=[
                dict(
                    label=f"Week {w}",
                    method="update",
                    args=[
                        {"visible": [i == w-1 for i in range(4)]},
                        {"title": f"{title} - {var} (Week {w})"}
                    ]
                )
                for w in range(1, 5)
            ]
        )],
        height=800
    )

    figs[var] = fig
    fig.show()  # Show each plot inline


# Save all figures into a single HTML with flex containers
html_blocks = []
for i, (name, fig) in enumerate(figs.items()):
    fig_html = pio.to_html(fig, include_plotlyjs=('cdn' if i == 0 else False), full_html=False)
    block = f"""
    <div style="display: flex; justify-content: center; margin-top: 20px;">
        <div id="{name.replace(' ', '_').lower()}_container" style="width: 90%;">
            {fig_html}
        </div>
    </div>
    """
    html_blocks.append(block)

# Combine all into one HTML document
full_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{data['title']} Plotly Plots</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.0/css/all.min.css">
    <link rel="stylesheet" href="/style.css">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
</head>
<body>
    <h1>{data['title']} ({dataset}) Plotly Plots</h1>

    <nav id="cut-nav"> 
        <ul id="cut-links">
            <li><a href="SRS.html" class="active">SRS</a></li>
            <li><a href="LPM.html">LPM</a></li>
            <li><a href="SPMF.html">SPMF</a></li>
            <li><a href="SONIC.html">SONIC</a></li>
        </ul>
    </nav>

    <!-- performance or selection plots -->
    <nav id="mode-nav">
    </nav>

    <!-- Section to display images -->
    <section>
        <h2 id="cut-title">{title} Description<br>
            Can drag to zoom in & double click to reset below images
        </h2>
        <div class="wrapper" id="image-container">
            <!-- Images will be dynamically added here -->
        </div>
    </section>

    <!-- Modal for Fullscreen Image -->
    <div id="image-modal" class="modal">
        <span class="close">&times;</span>
        <img class="modal-content" id="modal-img">
    </div>


    <!-- Sidebar -->
    <div id="sidebar">
    <h3>Test Plots</h3>
    <ul>
        <li><a href="../index.html#main" id="main-btn">main</a></li>
        <li><a href="SRS.html" id="test-btn">SRS</a></li>
        <li><a href="LPM.html" id="test-btn">LPM</a></li>
        <li><a href="SPMF.html" id="test-btn">SPMF</a></li>
        <li><a href="SONIC.html" id="test-btn">SONIC</a></li>
    </ul>
    </div>

    <div id="sidebar-toggle"><i class="fas fa-angle-right"></i></div>

    <!-- Plotly Graphs -->
  {''.join(html_blocks)}
    <script src="/sidebar.js"></script>


</body>
</html>
"""

# Save to file
with open("tmp1.html", "w", encoding="utf-8") as f:
    f.write(full_html)

In [85]:
import plotly.graph_objects as go
import pandas as pd
import os
from datetime import timedelta
from datetime import datetime


site = "C1"
instrument = "LPM"
fontfamily = "Open Sans"

import pandas as pd
import os
from datetime import timedelta

def load_and_prepare_dataframe(site: str, instrument: str, yml: dict, data_folder: str = "../../data") -> tuple[pd.DataFrame, list[str], str]:
    data_info = yml['NSA'][site][instrument]
    vars = data_info['var_name']
    title = data_info['title']
    filename = data_info['filename']

    # Load and resample
    df = pd.read_csv(os.path.join(data_folder, filename), usecols=vars)
    df[vars[0]] = pd.to_datetime(df[vars[0]])
    df = df.set_index(vars[0])
    df = df.resample("10T").mean()
    df = df.dropna(how='all', subset=vars[1:])

    # Define valid time window (last 4 full weeks)
    last_valid_time = df[vars[1:]].dropna(how='all').index.max()
    if last_valid_time is None:
        raise ValueError(f"No valid data found for {site}-{instrument}")
    
    end_of_week_4 = last_valid_time.floor("D") + pd.Timedelta(hours=23, minutes=50)
    start_of_week_1 = end_of_week_4 - timedelta(days=28)
    seconds_in_week = 7 * 24 * 60 * 60

    df = df.loc[start_of_week_1:end_of_week_4 - pd.Timedelta(minutes=9)].reset_index()
    df['week'] = (((df[vars[0]] - start_of_week_1).dt.total_seconds()) // seconds_in_week + 1).astype(int)
    df['is_nan'] = df[vars[1]].isna().astype(int)

    return df, vars, title

df, vars, title = load_and_prepare_dataframe("C1", "LPM", yml)

fig = go.Figure()

for week in range(1, 5):
    df_w = df[(df['week'] == week)]
    print(f"Week {week}: {len(df_w)} missing points")
    if not df_w.empty:
        fig.add_trace(go.Scattergl(
            x=df_w[vars[0]],
            y=df_w['is_nan'] + df_w['week'],
            mode='markers',
            marker=dict(symbol='square', size=6, opacity=0.8),
            name=f"Week {week}",
            showlegend=True
        ))

fig.update_layout(
    paper_bgcolor='#B5828C',
    plot_bgcolor='#B5828C',
    title=dict(
        text=f"{title} - Missing Data (Weeks 1–4)",
        font=dict(color='#EBFDFB', size=26, family=fontfamily)
    ),
    xaxis=dict(
        title=dict(text=vars[0], font=dict(color='#EBFDFB', size=20, family=fontfamily)),
        tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)
    ),
    yaxis=dict(
        title=dict(text="Week", font=dict(color='#EBFDFB', size=20, family=fontfamily)),
        tickmode='array',
        tickvals=[1, 2, 3, 4],
        ticktext=["Week 1", "Week 2", "Week 3", "Week 4"],
        tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily),
        range=[0.5, 4.5]
    ),
    legend=dict(
        x=1.02, y=1, yanchor='top',
        font=dict(color='#EBFDFB', size=15, family=fontfamily)
    ),
    margin=dict(r=100),
    height=500
)

fig.show()




Week 1: 1008 missing points
Week 2: 864 missing points
Week 3: 720 missing points
Week 4: 1008 missing points


In [104]:
from datetime import datetime

data_level = "a1"

snow_year = 9999
current_time = datetime.now()
if current_time.month < 8:
    snow_year = current_time.year
else:
    snow_year = current_time.year + 1
print(f"Snow Year: {snow_year}")

for loc in yml.keys():
    for site in yml['NSA'].keys():
        for instrument in yml['NSA'][site].keys():
            print(loc, site, instrument)
            datastream = f"{loc.lower()}{instrument.lower()}{site}.{data_level}"
            filename = f"{datastream}_snowyear_{snow_year}.csv"
            distance_var = f"distance_1_{datastream}_{snow_year}"
            #print(datastream, filename, distance_var)

datastream = f"{loc.lower()}lpmC1.{data_level}"

Snow Year: 2025
NSA C1 SRS
NSA C1 LPM
NSA C1 SPMF
NSA C1 SONIC
NSA E12 SRS
NSA E12 LPM
NSA E12 SPMF
NSA E12 SONIC
NSA E12 WBGEONOR
NSA E10 LPM


In [ ]:


site = "E12"
instrument = "SPMF"

df, vars, title = load_and_prepare_dataframe(site, instrument, yml)

figs = {}
for var in vars[1:]:

    fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3], shared_xaxes=True, vertical_spacing=0.05)
    trace_meta, hist_meta = [], []

    # var = vars[1]
    time_col = vars[0]
    weeks = sorted(df['week'].unique(), reverse=True)  # show Week 4 first in dropdown

    # Add one line plot per week
    for week in weeks:
        df_week = df[df['week'] == week]
        trace = go.Scattergl(
            x=df_week[time_col],
            y=df_week[var],
            mode='lines',
            name=f"{var} (Week {week})",
            visible=(week == weeks[0]),
            showlegend=True,
            line=dict(color="#55AD9B")
        )
        fig.add_trace(trace, row=1, col=1)
        trace_meta.append(week)


    for week in weeks:
        df_week = df[df['week'] == week]
        hist = go.Histogram(
            x=df_week[var],
            nbinsx=100,
            marker=dict(color="#55AD9B"),
            opacity=0.75,
            name=f"{var} Histogram (Week {week})",
            visible=(week == weeks[0]),
            showlegend=False
        )
        fig.add_trace(hist, row=2, col=1)
        hist_meta.append(week)
        
    # Create dropdown menu buttons
    buttons = []
    for i, week in enumerate(weeks):
        n_weeks = len(weeks)
        visibility = [j == i for j in range(n_weeks)] + [j == i for j in range(n_weeks)]
        buttons.append(dict(
            label=f"Week {week}",
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"{title} - {var} (Week {week})"}
            ]
        ))

    # Layout
    fig.update_layout(
        paper_bgcolor='#B5828C',
        title=dict(
            text=f"{title} - {var} (Week {weeks[0]})",
            font=dict(color='#EBFDFB', size=26, family=fontfamily)
        ),
        xaxis2=dict(  # x-axis of histogram (bottom plot)
            title=dict(text=time_col, font=dict(color='#EBFDFB', size=20, family=fontfamily)),
            tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)
        ),
        yaxis=dict(
            title=dict(text=var, font=dict(color='#EBFDFB', size=20, family=fontfamily)),
            tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)
        ),
        legend=dict(
            x=1.02, y=1, yanchor='top',
            font=dict(color='#EBFDFB', size=15, family=fontfamily)
        ),
        updatemenus=[dict(
            type="dropdown",
            direction="down",
            showactive=True,
            x=1.02,
            y=1.1,
            xanchor="left",
            yanchor="top",
            font=dict(color="lightgrey", size=15, family=fontfamily),
            buttons=buttons
        )],
        height=800
    )
    figs[var] = fig
    fig.show()

html_blocks, mode_blocks = [], []
for i, (name, fig) in enumerate(figs.items()):

    mode_block = f"""
            <li data-target="{name}_container"><a href="#">{name}</a></li>
    """
    mode_blocks.append(mode_block)


    fig_html = pio.to_html(fig, include_plotlyjs=False, full_html=False)
    html_block = f"""
    <div class="plot-wrapper" data-container="{name}_container">
        <div id="{name}_container"  class="plot-container">
            {fig_html}
        </div>
    </div>
    """
    html_blocks.append(html_block)

full_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title} Plotly Plots</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.0/css/all.min.css">
    <link rel="stylesheet" href="../../style.css">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
</head>
<body>
    <h1>{title} ({instrument}, {site}) Plotly Plots</h1>

    <nav id="cut-nav"> 
        <ul id="cut-links"></ul>
    </nav>

    <!-- performance or selection plots -->
    <nav id="mode-nav">
        <ul>
            <li>Plot: </li>
            {''.join(mode_blocks)}
        </ul>
    </nav>

    <!-- Section to display images -->
    <section>
        <h2 id="cut-title">{title} ({instrument}, {site}) Description<br>
            Can drag to zoom in & double click to reset below images
        </h2>
        <div class="wrapper" id="image-container">
            <!-- Images will be dynamically added here -->
        </div>
    </section>

    <!-- Modal for Fullscreen Image -->
    <div id="image-modal" class="modal">
        <span class="close">&times;</span>
        <img class="modal-content" id="modal-img">
    </div>

    {''.join(html_blocks)}

    <script src="../../cut_link.js"></script>
    <script src="../../scroll.js"></script>
    <script src="../../navigation.js"></script>
</body>
</html>
"""

with open(f'../tmp/{site}/{instrument}.html', "w", encoding="utf-8") as f:
    f.write(full_html)


In [53]:
import plotly.graph_objects as go
import pandas as pd
import os
from datetime import timedelta


site = "C1"
instrument = "LPM"
fontfamily = "Open Sans"

data_info = yml['NSA'][site][instrument]
vars = data_info['var_name']
title = data_info['title']
filename = data_info['filename']

df = pd.read_csv(os.path.join("../../data", filename), usecols=vars)
df[vars[0]] = pd.to_datetime(df[vars[0]])
df = df.set_index(vars[0])
df = df.resample("10T").mean()
df = df.dropna(how='all', subset=vars[1:])

last_valid_time = df[vars[1:]].dropna(how='all').index.max()
end_of_week_4 = last_valid_time.floor("D") + pd.Timedelta(hours=23, minutes=50)
start_of_week_1 = end_of_week_4 - timedelta(days=28)
seconds_in_week = 7 * 24 * 60 * 60

df = df.loc[start_of_week_1:end_of_week_4 - pd.Timedelta(minutes=9)].reset_index()
df['week'] = (((df[vars[0]] - start_of_week_1).dt.total_seconds()) // seconds_in_week + 1).astype(int)
df['is_nan'] = df[vars[1]].isna().astype(int)


fig = go.Figure()

for week in range(1, 5):
    df_w = df[(df['week'] == week)]
    print(f"Week {week}: {len(df_w)} missing points")
    if not df_w.empty:
        fig.add_trace(go.Scattergl(
            x=df_w[vars[0]],
            y=df_w['is_nan'] + df_w['week'],
            mode='markers',
            marker=dict(symbol='square', size=6, opacity=0.8),
            name=f"Week {week}",
            showlegend=True
        ))

fig.update_layout(
    paper_bgcolor='#B5828C',
    plot_bgcolor='#B5828C',
    title=dict(
        text=f"{title} - Missing Data (Weeks 1–4)",
        font=dict(color='#EBFDFB', size=26, family=fontfamily)
    ),
    xaxis=dict(
        title=dict(text=vars[0], font=dict(color='#EBFDFB', size=20, family=fontfamily)),
        tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily)
    ),
    yaxis=dict(
        title=dict(text="Week", font=dict(color='#EBFDFB', size=20, family=fontfamily)),
        tickmode='array',
        tickvals=[1, 2, 3, 4],
        ticktext=["Week 1", "Week 2", "Week 3", "Week 4"],
        tickfont=dict(color='#EBFDFB', size=13.5, family=fontfamily),
        range=[0.5, 4.5]
    ),
    legend=dict(
        x=1.02, y=1, yanchor='top',
        font=dict(color='#EBFDFB', size=15, family=fontfamily)
    ),
    margin=dict(r=100),
    height=500
)

fig.show()


Week 1: 1008 missing points
Week 2: 864 missing points
Week 3: 720 missing points
Week 4: 1008 missing points


In [ ]:
def load_and_prepare_dataframe(site: str, instrument: str, yml: dict, data_folder: str = "../../data") -> pd.DataFrame:
    data_info = yml['NSA'][site][instrument]
    vars = data_info['var_name']
    filename = data_info['filename']

    df = pd.read_csv(os.path.join(data_folder, filename), usecols=vars)
    df[vars[0]] = pd.to_datetime(df[vars[0]])
    df = df.set_index(vars[0])
    df = df.resample("10T").mean()
    df = df.dropna(how='all', subset=vars[1:])

    last_valid_time = df.dropna(how='all')[vars[1:]].last_valid_index()
    if last_valid_time is None:
        raise ValueError(f"No valid data found for {site}-{instrument}")
    first_valid_time = last_valid_time - timedelta(days=21)
    df = df.loc[first_valid_time:last_valid_time].reset_index()

    df['week'] = ((df[vars[0]] - df[vars[0]].min()).dt.days // 7 + 1).astype(int)
    return df, vars, data_info['title']


fontfamily = "Open Sans"
dataset = 'SPMF'

data = yml['NSA']['C1'][dataset]
vars = data['var_name']
title = data['title']
filename = data['filename']

df, vars, title = load_and_prepare_dataframe(site, instrument, yml)

